# Global RMSE and ACC for Ragasa and Yagi

This notebook computes latitude-weighted global RMSE and anomaly correlation coefficient (ACC) for five AI weather models against ERA5.

- Events: Ragasa (2025) and Yagi (2024)
- Models: Pangu, GraphCast, FengWu, FuXi, and Aurora
- Variables: U10, V10, MSLP, T2M, U850, V850, Z850, and T850
- Forecast leads: 6-72 h at 6-hour intervals
- Outputs: `./output/<event>_<model>_era5_<metric>.csv`

Only global metrics are calculated.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

INPUT_DIR = Path("./input")
OUTPUT_DIR = Path("./output")
AI_ROOT = Path("../../AI_forecasting_result")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_HOURS = np.arange(6, 73, 6)

VARIABLE_SPECS = [
    {"name": "U10", "layer": "surface", "era_var": "u10", "unit": "m s^-1"},
    {"name": "V10", "layer": "surface", "era_var": "v10", "unit": "m s^-1"},
    {"name": "MSLP", "layer": "surface", "era_var": "msl", "unit": "Pa"},
    {"name": "T2M", "layer": "surface", "era_var": "t2m", "unit": "K"},
    {"name": "U850", "layer": "upper", "era_var": "u", "unit": "m s^-1"},
    {"name": "V850", "layer": "upper", "era_var": "v", "unit": "m s^-1"},
    {"name": "Z850", "layer": "upper", "era_var": "z", "unit": "m^2 s^-2"},
    {"name": "T850", "layer": "upper", "era_var": "t", "unit": "K"},
]

EVENTS = {
    "ragasa": {
        "expected_times": pd.date_range("2025-09-22 01:00", periods=12, freq="6h"),
        "era_surface": INPUT_DIR / "era5_surface_2025092201-2025092419.nc",
        "era_upper": INPUT_DIR / "era5_850hPa_2025092201-2025092419.nc",
        "pangu_surface": AI_ROOT / "Ragasa/Pangu/output_48/2025-09-21-19-00to2025-09-24-19-00/surface_combined.nc",
        "pangu_upper": AI_ROOT / "Ragasa/Pangu/output_48/2025-09-21-19-00to2025-09-24-19-00/upper_combined.nc",
        "graphcast_dir": AI_ROOT / "Ragasa/Graphcast/output_48/2025092119",
        "fengwu_dir": AI_ROOT / "Ragasa/Fengwu/output_48/2025-09-21-13-00_to_2025-09-21-19-00",
        "fuxi_dir": AI_ROOT / "Ragasa/Fuxi/output_48/2025092113_to_2025092119",
        "fuxi_init": pd.Timestamp("2025-09-21 19:00"),
        "aurora_dir": AI_ROOT / "Ragasa/Aurora/output_48/WP/Ragasa/202509211300",
    },
    "yagi": {
        "expected_times": pd.date_range("2024-09-03 18:00", periods=12, freq="6h"),
        "era_surface": INPUT_DIR / "era5_surface_2024090318-2024090612.nc",
        "era_upper": INPUT_DIR / "era5_850hPa_2024090318-2024090612.nc",
        "pangu_surface_dir": AI_ROOT / "Yagi/Pangu/48",
        "pangu_upper_dir": AI_ROOT / "Yagi/Pangu/48",
        "graphcast_dir": AI_ROOT / "Yagi/Graphcast/48",
        "fengwu_dir": AI_ROOT / "Yagi/Fengwu/48",
        "fuxi_dir": AI_ROOT / "Yagi/Fuxi/48",
        "fuxi_init": pd.Timestamp("2024-09-03 12:00"),
        "aurora_dir": AI_ROOT / "Yagi/Aurora/48",
    },
}

MODEL_ORDER = ("pangu", "graphcast", "fengwu", "fuxi", "aurora")


In [ ]:
def _require_path(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path.resolve()}")
    return path


def _netcdf_files(directory, label):
    directory = _require_path(directory, label)
    files = sorted(directory.glob("*.nc"))
    if not files:
        raise FileNotFoundError(f"No NetCDF files found for {label}: {directory.resolve()}")
    return files


def _timestamp_from_dataset(path, coordinate, *, decode_timedelta=False):
    with xr.open_dataset(path, decode_timedelta=decode_timedelta) as dataset:
        if coordinate not in dataset.coords:
            raise KeyError(f"{path.name} does not contain time coordinate {coordinate!r}")
        values = np.asarray(dataset[coordinate].values).reshape(-1)
        if values.size != 1:
            raise ValueError(
                f"Expected one {coordinate} value in {path.name}, found {values.size}"
            )
        return pd.Timestamp(values[0])


def _select_expected_times(dataset, expected_times, label):
    if "time" in dataset.dims and "valid_time" not in dataset.dims:
        dataset = dataset.rename({"time": "valid_time"})
    if "valid_time" not in dataset.coords:
        raise KeyError(f"{label} does not contain a valid_time coordinate")

    available_times = pd.DatetimeIndex(pd.to_datetime(dataset["valid_time"].values))
    if available_times.has_duplicates:
        duplicates = available_times[available_times.duplicated()].strftime("%Y%m%d%H").tolist()
        raise ValueError(f"Duplicate valid times in {label}: {duplicates}")

    missing_times = expected_times.difference(available_times)
    if len(missing_times):
        missing_text = missing_times.strftime("%Y%m%d%H").tolist()
        raise ValueError(f"Missing expected valid times in {label}: {missing_text}")

    selected = dataset.sel(valid_time=expected_times.values)
    selected_times = pd.DatetimeIndex(pd.to_datetime(selected["valid_time"].values))
    if not np.array_equal(selected_times.values, expected_times.values):
        raise ValueError(f"Unexpected valid-time order in {label}")
    return selected


def _normalise_grid(dataset, target_latitude, target_longitude):
    rename_map = {}
    if "lat" in dataset.dims:
        rename_map["lat"] = "latitude"
    if "lon" in dataset.dims:
        rename_map["lon"] = "longitude"
    if rename_map:
        dataset = dataset.rename(rename_map)

    required_dims = {"latitude", "longitude"}
    if not required_dims.issubset(dataset.dims):
        raise ValueError(
            f"Expected latitude/longitude dimensions, found {tuple(dataset.dims)}"
        )

    dataset = dataset.sortby("latitude", ascending=False)
    dataset = dataset.sortby("longitude", ascending=True)
    return dataset.reindex(latitude=target_latitude, longitude=target_longitude)


def _drop_auxiliary_coordinates(dataset):
    auxiliary = [
        name
        for name in ("batch", "time", "datetime", "step")
        if name in dataset.coords and name not in dataset.dims
    ]
    if auxiliary:
        dataset = dataset.drop_vars(auxiliary, errors="ignore")
    return dataset


def _concat_time_parts(parts, label):
    if not parts:
        raise ValueError(f"No forecast fields loaded for {label}")
    combined = xr.concat(
        parts,
        dim="valid_time",
        data_vars="all",
        coords="minimal",
        compat="override",
        join="exact",
    )
    return combined.sortby("valid_time")


def open_era5(event_name, config):
    surface_path = _require_path(config["era_surface"], f"{event_name} ERA5 surface file")
    upper_path = _require_path(config["era_upper"], f"{event_name} ERA5 850-hPa file")

    surface = xr.open_dataset(surface_path)
    upper = xr.open_dataset(upper_path)
    expected_times = config["expected_times"]

    try:
        surface = _select_expected_times(surface, expected_times, f"{event_name} ERA5 surface")
        upper = _select_expected_times(upper, expected_times, f"{event_name} ERA5 upper")

        missing_surface = {"u10", "v10", "msl", "t2m"}.difference(surface.data_vars)
        missing_upper = {"u", "v", "z", "t"}.difference(upper.data_vars)
        if missing_surface:
            raise KeyError(f"Missing ERA5 surface variables: {sorted(missing_surface)}")
        if missing_upper:
            raise KeyError(f"Missing ERA5 upper variables: {sorted(missing_upper)}")

        if "pressure_level" not in upper.coords:
            raise KeyError("ERA5 upper-air data does not contain pressure_level")
        pressure_levels = np.asarray(upper["pressure_level"].values, dtype=float)
        if not np.any(np.isclose(pressure_levels, 850.0)):
            raise ValueError(f"ERA5 upper-air data does not contain 850 hPa: {pressure_levels}")

        upper = upper.sel(pressure_level=850, drop=True)
        surface = surface.sortby("latitude", ascending=False).sortby("longitude")
        upper = upper.sortby("latitude", ascending=False).sortby("longitude")

        if not np.array_equal(surface["latitude"].values, upper["latitude"].values):
            raise ValueError(f"{event_name} ERA5 surface/upper latitude grids differ")
        if not np.array_equal(surface["longitude"].values, upper["longitude"].values):
            raise ValueError(f"{event_name} ERA5 surface/upper longitude grids differ")
    except Exception:
        surface.close()
        upper.close()
        raise

    return surface, upper


def weighted_rmse(forecast_field, reference_field, latitude_weights):
    weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
    valid_mask = np.isfinite(forecast_field) & np.isfinite(reference_field)
    if not np.any(valid_mask):
        return np.nan

    weights_valid = weights_2d[valid_mask]
    squared_error = (forecast_field[valid_mask] - reference_field[valid_mask]) ** 2
    weight_sum = np.sum(weights_valid)
    if weight_sum == 0:
        return np.nan
    return np.sqrt(np.sum(weights_valid * squared_error) / weight_sum)


def weighted_acc(forecast_field, reference_field, climatology_field, latitude_weights):
    forecast_anomaly = forecast_field - climatology_field
    reference_anomaly = reference_field - climatology_field
    weights_2d = np.broadcast_to(latitude_weights[:, None], forecast_field.shape)
    valid_mask = np.isfinite(forecast_anomaly) & np.isfinite(reference_anomaly)
    if not np.any(valid_mask):
        return np.nan

    weights_valid = weights_2d[valid_mask]
    forecast_valid = forecast_anomaly[valid_mask]
    reference_valid = reference_anomaly[valid_mask]
    denominator = np.sqrt(
        np.sum(weights_valid * forecast_valid**2)
        * np.sum(weights_valid * reference_valid**2)
    )
    if denominator == 0:
        return np.nan
    numerator = np.sum(weights_valid * forecast_valid * reference_valid)
    return numerator / denominator


def validate_model_dataset(dataset, event_name, model_name, config, era_surface):
    expected_variables = [spec["name"] for spec in VARIABLE_SPECS]
    missing_variables = set(expected_variables).difference(dataset.data_vars)
    if missing_variables:
        raise KeyError(
            f"Missing variables for {event_name}/{model_name}: {sorted(missing_variables)}"
        )

    expected_times = config["expected_times"]
    actual_times = pd.DatetimeIndex(pd.to_datetime(dataset["valid_time"].values))
    if not np.array_equal(actual_times.values, expected_times.values):
        raise ValueError(
            f"Unexpected valid times for {event_name}/{model_name}: "
            f"{actual_times.strftime('%Y%m%d%H').tolist()}"
        )

    if not np.array_equal(dataset["latitude"].values, era_surface["latitude"].values):
        raise ValueError(f"Latitude grid mismatch for {event_name}/{model_name}")
    if not np.array_equal(dataset["longitude"].values, era_surface["longitude"].values):
        raise ValueError(f"Longitude grid mismatch for {event_name}/{model_name}")

    for variable_name in expected_variables:
        field = dataset[variable_name].transpose("valid_time", "latitude", "longitude")
        if field.sizes["valid_time"] != len(expected_times):
            raise ValueError(f"Unexpected time count for {event_name}/{model_name}/{variable_name}")
        valid_by_time = np.asarray(
            field.notnull().any(dim=("latitude", "longitude")).values,
            dtype=bool,
        )
        if not np.all(valid_by_time):
            bad_indices = np.flatnonzero(~valid_by_time).tolist()
            raise ValueError(
                f"All-NaN fields for {event_name}/{model_name}/{variable_name} "
                f"at time indices {bad_indices}"
            )


def compute_and_save_metrics(event_name, model_name, forecast, era_surface, era_upper, config):
    base_table = pd.DataFrame(
        {
            "forecast_hour": FORECAST_HOURS,
            "valid_time": config["expected_times"].strftime("%Y%m%d%H"),
        }
    )
    rmse_table = base_table.copy()
    acc_table = base_table.copy()

    latitude_weights = np.cos(np.deg2rad(era_surface["latitude"].values))

    for spec in VARIABLE_SPECS:
        forecast_da = forecast[spec["name"]].transpose(
            "valid_time", "latitude", "longitude"
        )
        reference_source = era_surface if spec["layer"] == "surface" else era_upper
        reference_da = reference_source[spec["era_var"]].transpose(
            "valid_time", "latitude", "longitude"
        )
        climatology = reference_da.mean(dim="valid_time").values

        rmse_values = []
        acc_values = []
        for time_index in range(len(config["expected_times"])):
            forecast_field = forecast_da.isel(valid_time=time_index).values
            reference_field = reference_da.isel(valid_time=time_index).values
            rmse_values.append(
                weighted_rmse(forecast_field, reference_field, latitude_weights)
            )
            acc_values.append(
                weighted_acc(
                    forecast_field,
                    reference_field,
                    climatology,
                    latitude_weights,
                )
            )

        rmse_array = np.asarray(rmse_values, dtype=float)
        acc_array = np.asarray(acc_values, dtype=float)
        if not np.all(np.isfinite(rmse_array)):
            raise ValueError(f"Non-finite RMSE for {event_name}/{model_name}/{spec['name']}")
        if np.any(rmse_array < 0):
            raise ValueError(f"Negative RMSE for {event_name}/{model_name}/{spec['name']}")
        if not np.all(np.isfinite(acc_array)):
            raise ValueError(f"Non-finite ACC for {event_name}/{model_name}/{spec['name']}")
        if np.any((acc_array < -1.000001) | (acc_array > 1.000001)):
            raise ValueError(f"ACC outside [-1, 1] for {event_name}/{model_name}/{spec['name']}")

        rmse_table[spec["name"]] = rmse_array
        acc_table[spec["name"]] = acc_array

    rmse_path = OUTPUT_DIR / f"{event_name}_{model_name}_era5_rmse.csv"
    acc_path = OUTPUT_DIR / f"{event_name}_{model_name}_era5_acc.csv"
    rmse_table.to_csv(rmse_path, index=False)
    acc_table.to_csv(acc_path, index=False)

    print(f"Saved {rmse_path.resolve()}")
    print(f"Saved {acc_path.resolve()}")
    return rmse_table, acc_table


In [ ]:
PANGU_SURFACE_VARIABLES = {
    "U10": "u10",
    "V10": "v10",
    "MSLP": "msl",
    "T2M": "t2m",
}
PANGU_UPPER_VARIABLES = {
    "U850": "u",
    "V850": "v",
    "Z850": "z",
    "T850": "t",
}

GRAPHCAST_SURFACE_VARIABLES = {
    "U10": "10m_u_component_of_wind",
    "V10": "10m_v_component_of_wind",
    "MSLP": "mean_sea_level_pressure",
    "T2M": "2m_temperature",
}
GRAPHCAST_UPPER_VARIABLES = {
    "U850": "u_component_of_wind",
    "V850": "v_component_of_wind",
    "Z850": "geopotential",
    "T850": "temperature",
}

FENGWU_VARIABLES = {
    "U10": "u10",
    "V10": "v10",
    "MSLP": "msl",
    "T2M": "t2m",
    "U850": "u850",
    "V850": "v850",
    "Z850": "z850",
    "T850": "t850",
}

FUXI_LEVELS = {
    "U10": "U10",
    "V10": "V10",
    "MSLP": "MSL",
    "T2M": "T2M",
    "U850": "U850",
    "V850": "V850",
    "Z850": "Z850",
    "T850": "T850",
}

AURORA_SURFACE_VARIABLES = {
    "U10": "u10",
    "V10": "v10",
    "MSLP": "msl",
    "T2M": "t2m",
}
AURORA_UPPER_VARIABLES = {
    "U850": "u",
    "V850": "v",
    "Z850": "z",
    "T850": "t",
}


def _pangu_dataset(surface, upper):
    missing_surface = set(PANGU_SURFACE_VARIABLES.values()).difference(surface.data_vars)
    missing_upper = set(PANGU_UPPER_VARIABLES.values()).difference(upper.data_vars)
    if missing_surface or missing_upper:
        raise KeyError(
            f"Missing Pangu variables: surface={sorted(missing_surface)}, "
            f"upper={sorted(missing_upper)}"
        )
    if "level" not in upper.coords or 3 not in upper["level"].values:
        raise ValueError("Pangu upper-air data does not contain internal level 3 (850 hPa)")

    data_vars = {
        canonical: surface[source]
        for canonical, source in PANGU_SURFACE_VARIABLES.items()
    }
    data_vars.update(
        {
            canonical: upper[source].sel(level=3, drop=True)
            for canonical, source in PANGU_UPPER_VARIABLES.items()
        }
    )
    return xr.Dataset(data_vars)


def load_pangu(config, target_latitude, target_longitude):
    if "pangu_surface" in config:
        surface_path = _require_path(config["pangu_surface"], "Pangu surface file")
        upper_path = _require_path(config["pangu_upper"], "Pangu upper-air file")
        with xr.open_dataset(surface_path) as surface, xr.open_dataset(upper_path) as upper:
            dataset = _pangu_dataset(surface, upper)
            dataset = _select_expected_times(dataset, config["expected_times"], "Pangu")
            dataset = _normalise_grid(dataset, target_latitude, target_longitude)
            return dataset.load()

    surface_files = sorted(
        _require_path(config["pangu_surface_dir"], "Pangu surface directory").glob(
            "surface_*.nc"
        )
    )
    upper_files = sorted(
        _require_path(config["pangu_upper_dir"], "Pangu upper-air directory").glob(
            "upper_*.nc"
        )
    )
    if not surface_files or not upper_files:
        raise FileNotFoundError("Pangu surface/upper forecast files are missing")

    surface_by_time = {
        _timestamp_from_dataset(path, "valid_time"): path for path in surface_files
    }
    upper_by_time = {
        _timestamp_from_dataset(path, "valid_time"): path for path in upper_files
    }
    if set(surface_by_time) != set(upper_by_time):
        raise ValueError("Pangu surface and upper-air valid times differ")

    parts = []
    for valid_time in sorted(surface_by_time):
        with xr.open_dataset(surface_by_time[valid_time]) as surface, xr.open_dataset(
            upper_by_time[valid_time]
        ) as upper:
            part = _pangu_dataset(surface, upper)
            part = _normalise_grid(part, target_latitude, target_longitude).load()
            parts.append(part)

    dataset = _concat_time_parts(parts, "Pangu")
    return _select_expected_times(dataset, config["expected_times"], "Pangu")


def load_graphcast(config, target_latitude, target_longitude):
    files = _netcdf_files(config["graphcast_dir"], "GraphCast directory")
    parts = []

    for path in files:
        with xr.open_dataset(path, decode_timedelta=False) as source:
            if "datetime" not in source.coords:
                raise KeyError(f"{path.name} does not contain datetime")
            valid_time = pd.Timestamp(np.asarray(source["datetime"].values).reshape(-1)[0])

            data_vars = {}
            for canonical, variable_name in GRAPHCAST_SURFACE_VARIABLES.items():
                field = source[variable_name]
                indexers = {
                    dim: 0 for dim in ("batch", "time") if dim in field.dims
                }
                data_vars[canonical] = field.isel(indexers, drop=True)

            for canonical, variable_name in GRAPHCAST_UPPER_VARIABLES.items():
                field = source[variable_name]
                indexers = {
                    dim: 0 for dim in ("batch", "time") if dim in field.dims
                }
                field = field.isel(indexers, drop=True)
                data_vars[canonical] = field.sel(level=850, drop=True)

            part = xr.Dataset(data_vars)
            part = _drop_auxiliary_coordinates(part)
            part = part.expand_dims(valid_time=[valid_time])
            part = _normalise_grid(part, target_latitude, target_longitude).load()
            parts.append(part)

    dataset = _concat_time_parts(parts, "GraphCast")
    return _select_expected_times(dataset, config["expected_times"], "GraphCast")


def load_fengwu(config, target_latitude, target_longitude):
    files = _netcdf_files(config["fengwu_dir"], "FengWu directory")
    parts = []

    for path in files:
        with xr.open_dataset(path) as source:
            if "valid_time" not in source.coords:
                raise KeyError(f"{path.name} does not contain valid_time")
            valid_values = np.asarray(source["valid_time"].values).reshape(-1)
            if valid_values.size != 1:
                raise ValueError(f"Expected one valid_time in {path.name}")
            missing = set(FENGWU_VARIABLES.values()).difference(source.data_vars)
            if missing:
                raise KeyError(f"Missing FengWu variables in {path.name}: {sorted(missing)}")

            part = xr.Dataset(
                {
                    canonical: source[variable_name]
                    for canonical, variable_name in FENGWU_VARIABLES.items()
                }
            )
            part = _normalise_grid(part, target_latitude, target_longitude).load()
            parts.append(part)

    dataset = _concat_time_parts(parts, "FengWu")
    return _select_expected_times(dataset, config["expected_times"], "FengWu")


def load_fuxi(config, target_latitude, target_longitude):
    files = _netcdf_files(config["fuxi_dir"], "FuXi directory")
    parts = []

    for path in files:
        with xr.open_dataset(path, decode_timedelta=False) as source:
            if "__xarray_dataarray_variable__" not in source.data_vars:
                raise KeyError(f"Unexpected FuXi data variable in {path.name}")
            if "step" not in source.coords:
                raise KeyError(f"{path.name} does not contain step")
            step_values = np.asarray(source["step"].values).reshape(-1)
            if step_values.size != 1:
                raise ValueError(f"Expected one forecast step in {path.name}")
            step_hours = int(step_values[0])
            valid_time = config["fuxi_init"] + pd.Timedelta(hours=step_hours)

            source_field = source["__xarray_dataarray_variable__"]
            available_levels = set(map(str, source["level"].values.tolist()))
            missing_levels = set(FUXI_LEVELS.values()).difference(available_levels)
            if missing_levels:
                raise KeyError(f"Missing FuXi levels in {path.name}: {sorted(missing_levels)}")

            data_vars = {}
            for canonical, level_name in FUXI_LEVELS.items():
                field = source_field.sel(level=level_name, drop=True)
                indexers = {
                    dim: 0 for dim in ("time", "step") if dim in field.dims
                }
                data_vars[canonical] = field.isel(indexers, drop=True)

            part = xr.Dataset(data_vars)
            part = _drop_auxiliary_coordinates(part)
            part = part.expand_dims(valid_time=[valid_time])
            part = _normalise_grid(part, target_latitude, target_longitude).load()
            parts.append(part)

    dataset = _concat_time_parts(parts, "FuXi")
    return _select_expected_times(dataset, config["expected_times"], "FuXi")


def load_aurora(config, target_latitude, target_longitude):
    files = _netcdf_files(config["aurora_dir"], "Aurora directory")
    parts = []

    for path in files:
        with xr.open_dataset(path, decode_timedelta=False) as source:
            if "time" not in source.coords:
                raise KeyError(f"{path.name} does not contain time")
            time_values = np.asarray(source["time"].values).reshape(-1)
            if time_values.size != 1:
                raise ValueError(f"Expected one time value in {path.name}")
            valid_time = pd.Timestamp(time_values[0])

            data_vars = {
                canonical: source[variable_name]
                for canonical, variable_name in AURORA_SURFACE_VARIABLES.items()
            }
            data_vars.update(
                {
                    canonical: source[variable_name].sel(level=850, drop=True)
                    for canonical, variable_name in AURORA_UPPER_VARIABLES.items()
                }
            )

            part = xr.Dataset(data_vars)
            part = _drop_auxiliary_coordinates(part)
            part = part.expand_dims(valid_time=[valid_time])
            part = _normalise_grid(part, target_latitude, target_longitude).load()
            parts.append(part)

    dataset = _concat_time_parts(parts, "Aurora")
    return _select_expected_times(dataset, config["expected_times"], "Aurora")


MODEL_LOADERS = {
    "pangu": load_pangu,
    "graphcast": load_graphcast,
    "fengwu": load_fengwu,
    "fuxi": load_fuxi,
    "aurora": load_aurora,
}


In [ ]:
results = {}

for event_name, config in EVENTS.items():
    print(f"\n{'=' * 72}\nEvent: {event_name}\n{'=' * 72}")
    era_surface, era_upper = open_era5(event_name, config)
    try:
        target_latitude = era_surface["latitude"]
        target_longitude = era_surface["longitude"]
        results[event_name] = {}

        for model_name in MODEL_ORDER:
            print(f"\nLoading and evaluating {event_name}/{model_name} ...")
            forecast = MODEL_LOADERS[model_name](
                config,
                target_latitude,
                target_longitude,
            )
            validate_model_dataset(
                forecast,
                event_name,
                model_name,
                config,
                era_surface,
            )
            results[event_name][model_name] = compute_and_save_metrics(
                event_name,
                model_name,
                forecast,
                era_surface,
                era_upper,
                config,
            )
            forecast.close()
            del forecast
    finally:
        era_surface.close()
        era_upper.close()

expected_output_names = {
    f"{event_name}_{model_name}_era5_{metric}.csv"
    for event_name in EVENTS
    for model_name in MODEL_ORDER
    for metric in ("rmse", "acc")
}
missing_outputs = sorted(
    name for name in expected_output_names if not (OUTPUT_DIR / name).exists()
)
if missing_outputs:
    raise RuntimeError(f"Missing output files: {missing_outputs}")

print(f"\nCompleted successfully: {len(expected_output_names)} global CSV files.")
